# 🔧 Módulo 10: El Constructor de Lenguajes — Mini Compilador
## Notebook de Conocimiento Guiado

**Objetivo:** Implementar un compilador completo (Lexer → Parser → AST → Intérprete)
para un mini-lenguaje de programación, sin dependencias externas.

**Lore:** Eres el Maestro Constructor del lenguaje de los robots galácticos.
Tu compilador convierte código fuente en instrucciones ejecutables.

| Sección | Concepto |
|---------|---------|
| 📚 Teoría | Fases de compilación, AST, gramáticas |
| 🔨 Demo | Lexer: código → tokens |
| 🔨 Demo | Parser recursivo descendente → AST |
| 🔨 Demo | Tree-walking interpreter |
| ✏️ Ejercicio 1 | Añadir soporte para `while` |
| ✏️ Ejercicio 2 | Añadir funciones `def` |
| 🎯 Quiz | Compiladores, parsers, tipos |

## 📚 Parte 1: Las Fases de Compilación

```
Código fuente (string)
       ↓
    LEXER       → secuencia de Tokens
       ↓
    PARSER      → Árbol de Sintaxis Abstracta (AST)
       ↓
  INTÉRPRETE    → resultado de ejecución
  (o GENERADOR
   DE CÓDIGO)
```

**Nuestro lenguaje soporta:**
```
let x = 5
let y = x + 3 * 2
if x > 4
    print(y)
end
```

**Gramática (BNF simplificada):**
```
programa   := sentencia*
sentencia  := let_stmt | if_stmt | print_stmt | expr
let_stmt   := "let" IDENT "=" expr
if_stmt    := "if" expr NEWLINE sentencia* "end"
print_stmt := "print" "(" expr ")"
expr       := comparacion
comparacion:= suma (("==" | "!=" | "<" | ">") suma)*
suma       := termino (("+"|"-") termino)*
termino    := factor (("*"|"/") factor)*
factor     := NUMBER | IDENT | "(" expr ")"
```

In [ ]:
# 🔨 IMPLEMENTACIÓN: Lexer (Analizador Léxico)

from enum import Enum, auto
from dataclasses import dataclass
from typing import List, Optional

class TipoToken(Enum):
    # Literales
    NUMERO = auto()
    BOOLEANO = auto()
    IDENTIFICADOR = auto()
    # Operadores
    MAS = auto()
    MENOS = auto()
    ESTRELLA = auto()
    BARRA = auto()
    ASIGNAR = auto()
    IGUAL = auto()
    NO_IGUAL = auto()
    MENOR = auto()
    MAYOR = auto()
    # Delimitadores
    PAREN_IZQ = auto()
    PAREN_DER = auto()
    COMA = auto()
    NUEVA_LINEA = auto()
    # Palabras clave
    LET = auto()
    IF = auto()
    ELSE = auto()
    END = auto()
    PRINT = auto()
    TRUE = auto()
    FALSE = auto()
    EOF = auto()

PALABRAS_CLAVE = {
    "let": TipoToken.LET, "if": TipoToken.IF,
    "else": TipoToken.ELSE, "end": TipoToken.END,
    "print": TipoToken.PRINT, "true": TipoToken.TRUE,
    "false": TipoToken.FALSE,
}

@dataclass
class Token:
    tipo: TipoToken
    valor: object
    linea: int = 0
    
    def __repr__(self):
        return f"Token({self.tipo.name}, {self.valor!r})"

class Lexer:
    def __init__(self, fuente: str):
        self.fuente = fuente
        self.pos = 0
        self.linea = 1
    
    def _actual(self): return self.fuente[self.pos] if self.pos < len(self.fuente) else None
    def _avanzar(self):
        c = self.fuente[self.pos]; self.pos += 1
        if c == '\n': self.linea += 1
        return c
    
    def tokenizar(self) -> List[Token]:
        tokens = []
        while self.pos < len(self.fuente):
            c = self._actual()
            
            if c in ' \t\r': self._avanzar()
            elif c == '\n':
                tokens.append(Token(TipoToken.NUEVA_LINEA, '\n', self.linea))
                self._avanzar()
            elif c == '#':
                while self._actual() and self._actual() != '\n': self._avanzar()
            elif c.isdigit():
                inicio = self.pos
                while self._actual() and (self._actual().isdigit() or self._actual() == '.'):
                    self._avanzar()
                tokens.append(Token(TipoToken.NUMERO, float(self.fuente[inicio:self.pos]), self.linea))
            elif c.isalpha() or c == '_':
                inicio = self.pos
                while self._actual() and (self._actual().isalnum() or self._actual() == '_'):
                    self._avanzar()
                palabra = self.fuente[inicio:self.pos]
                tipo = PALABRAS_CLAVE.get(palabra, TipoToken.IDENTIFICADOR)
                tokens.append(Token(tipo, palabra, self.linea))
            elif c == '=':
                self._avanzar()
                if self._actual() == '=': self._avanzar(); tokens.append(Token(TipoToken.IGUAL, '==', self.linea))
                else: tokens.append(Token(TipoToken.ASIGNAR, '=', self.linea))
            elif c == '!':
                self._avanzar()
                if self._actual() == '=': self._avanzar(); tokens.append(Token(TipoToken.NO_IGUAL, '!=', self.linea))
            elif c == '<': self._avanzar(); tokens.append(Token(TipoToken.MENOR, '<', self.linea))
            elif c == '>': self._avanzar(); tokens.append(Token(TipoToken.MAYOR, '>', self.linea))
            elif c == '+': self._avanzar(); tokens.append(Token(TipoToken.MAS, '+', self.linea))
            elif c == '-': self._avanzar(); tokens.append(Token(TipoToken.MENOS, '-', self.linea))
            elif c == '*': self._avanzar(); tokens.append(Token(TipoToken.ESTRELLA, '*', self.linea))
            elif c == '/': self._avanzar(); tokens.append(Token(TipoToken.BARRA, '/', self.linea))
            elif c == '(': self._avanzar(); tokens.append(Token(TipoToken.PAREN_IZQ, '(', self.linea))
            elif c == ')': self._avanzar(); tokens.append(Token(TipoToken.PAREN_DER, ')', self.linea))
            elif c == ',': self._avanzar(); tokens.append(Token(TipoToken.COMA, ',', self.linea))
            else: raise SyntaxError(f"Carácter inesperado: {c!r} en línea {self.linea}")
        
        tokens.append(Token(TipoToken.EOF, None, self.linea))
        return tokens

# Test
codigo = """
let x = 5
let y = x + 3 * 2
if x > 4
    print(y)
end
"""
lexer = Lexer(codigo)
tokens = lexer.tokenizar()
print("Tokens generados:")
for t in tokens:
    if t.tipo not in (TipoToken.NUEVA_LINEA, TipoToken.EOF):
        print(f"  {t}")

In [ ]:
# 🔨 IMPLEMENTACIÓN: AST Nodes

@dataclass
class NodoNumero:
    valor: float

@dataclass
class NodoBooleano:
    valor: bool

@dataclass
class NodoIdentificador:
    nombre: str

@dataclass
class NodoBinario:
    operador: str
    izquierda: object
    derecha: object

@dataclass
class NodoLet:
    nombre: str
    valor: object

@dataclass
class NodoIf:
    condicion: object
    then_cuerpo: list
    else_cuerpo: list

@dataclass
class NodoPrint:
    argumentos: list

# ─────────────────────────────────────────────────────────────────────────────
# PARSER: Tokens → AST
# ─────────────────────────────────────────────────────────────────────────────

class Parser:
    def __init__(self, tokens: List[Token]):
        self.tokens = [t for t in tokens if t.tipo != TipoToken.NUEVA_LINEA
                       or tokens.index(t) == 0]  # simplificación
        self.tokens = tokens
        self.pos = 0
    
    def _actual(self): return self.tokens[self.pos]
    def _consumir(self, tipo=None):
        t = self.tokens[self.pos]
        if tipo and t.tipo != tipo:
            raise SyntaxError(f"Esperaba {tipo.name}, obtuve {t.tipo.name} en línea {t.linea}")
        self.pos += 1
        return t
    def _saltar_nuevas_lineas(self):
        while self._actual().tipo == TipoToken.NUEVA_LINEA:
            self._consumir()
    
    def parsear(self):
        sentencias = []
        self._saltar_nuevas_lineas()
        while self._actual().tipo != TipoToken.EOF:
            sentencias.append(self._sentencia())
            self._saltar_nuevas_lineas()
        return sentencias
    
    def _sentencia(self):
        t = self._actual()
        if t.tipo == TipoToken.LET:
            return self._let_stmt()
        elif t.tipo == TipoToken.IF:
            return self._if_stmt()
        elif t.tipo == TipoToken.PRINT:
            return self._print_stmt()
        return self._expresion()
    
    def _let_stmt(self):
        self._consumir(TipoToken.LET)
        nombre = self._consumir(TipoToken.IDENTIFICADOR).valor
        self._consumir(TipoToken.ASIGNAR)
        valor = self._expresion()
        return NodoLet(nombre, valor)
    
    def _if_stmt(self):
        self._consumir(TipoToken.IF)
        condicion = self._expresion()
        self._saltar_nuevas_lineas()
        then_cuerpo = []
        while self._actual().tipo not in (TipoToken.END, TipoToken.ELSE, TipoToken.EOF):
            then_cuerpo.append(self._sentencia())
            self._saltar_nuevas_lineas()
        else_cuerpo = []
        if self._actual().tipo == TipoToken.ELSE:
            self._consumir(TipoToken.ELSE)
            self._saltar_nuevas_lineas()
            while self._actual().tipo not in (TipoToken.END, TipoToken.EOF):
                else_cuerpo.append(self._sentencia())
                self._saltar_nuevas_lineas()
        self._consumir(TipoToken.END)
        return NodoIf(condicion, then_cuerpo, else_cuerpo)
    
    def _print_stmt(self):
        self._consumir(TipoToken.PRINT)
        self._consumir(TipoToken.PAREN_IZQ)
        args = [self._expresion()]
        while self._actual().tipo == TipoToken.COMA:
            self._consumir()
            args.append(self._expresion())
        self._consumir(TipoToken.PAREN_DER)
        return NodoPrint(args)
    
    def _expresion(self): return self._comparacion()
    
    def _comparacion(self):
        izq = self._suma()
        while self._actual().tipo in (TipoToken.IGUAL, TipoToken.NO_IGUAL,
                                       TipoToken.MENOR, TipoToken.MAYOR):
            op = self._consumir().valor
            izq = NodoBinario(op, izq, self._suma())
        return izq
    
    def _suma(self):
        izq = self._termino()
        while self._actual().tipo in (TipoToken.MAS, TipoToken.MENOS):
            op = self._consumir().valor
            izq = NodoBinario(op, izq, self._termino())
        return izq
    
    def _termino(self):
        izq = self._factor()
        while self._actual().tipo in (TipoToken.ESTRELLA, TipoToken.BARRA):
            op = self._consumir().valor
            izq = NodoBinario(op, izq, self._factor())
        return izq
    
    def _factor(self):
        t = self._actual()
        if t.tipo == TipoToken.NUMERO:
            self._consumir()
            return NodoNumero(t.valor)
        if t.tipo in (TipoToken.TRUE, TipoToken.FALSE):
            self._consumir()
            return NodoBooleano(t.tipo == TipoToken.TRUE)
        if t.tipo == TipoToken.IDENTIFICADOR:
            self._consumir()
            return NodoIdentificador(t.valor)
        if t.tipo == TipoToken.PAREN_IZQ:
            self._consumir()
            expr = self._expresion()
            self._consumir(TipoToken.PAREN_DER)
            return expr
        raise SyntaxError(f"Token inesperado: {t}")

# Test del parser
tokens = Lexer(codigo).tokenizar()
parser = Parser(tokens)
ast = parser.parsear()
print("AST generado:")
for nodo in ast:
    print(f"  {nodo}")

In [ ]:
# 🔨 IMPLEMENTACIÓN: Tree-Walking Interpreter

class Interprete:
    def __init__(self):
        self.env = {}           # Variables: {nombre: valor}
        self.salida = []        # Output del print
    
    def ejecutar(self, sentencias):
        for stmt in sentencias:
            self._evaluar(stmt)
    
    def _evaluar(self, nodo):
        if isinstance(nodo, NodoNumero):
            return nodo.valor
        if isinstance(nodo, NodoBooleano):
            return nodo.valor
        if isinstance(nodo, NodoIdentificador):
            if nodo.nombre not in self.env:
                raise NameError(f"Variable no definida: {nodo.nombre!r}")
            return self.env[nodo.nombre]
        if isinstance(nodo, NodoBinario):
            izq = self._evaluar(nodo.izquierda)
            der = self._evaluar(nodo.derecha)
            ops = {'+': izq+der, '-': izq-der, '*': izq*der,
                   '/': izq/der, '==': izq==der, '!=': izq!=der,
                   '<': izq<der, '>': izq>der}
            return ops[nodo.operador]
        if isinstance(nodo, NodoLet):
            self.env[nodo.nombre] = self._evaluar(nodo.valor)
            return None
        if isinstance(nodo, NodoIf):
            if self._evaluar(nodo.condicion):
                for s in nodo.then_cuerpo: self._evaluar(s)
            elif nodo.else_cuerpo:
                for s in nodo.else_cuerpo: self._evaluar(s)
            return None
        if isinstance(nodo, NodoPrint):
            vals = [str(int(self._evaluar(a)) if isinstance(self._evaluar(a), float)
                        and self._evaluar(a) == int(self._evaluar(a))
                        else self._evaluar(a)) for a in nodo.argumentos]
            resultado = " ".join(vals)
            self.salida.append(resultado)
            print(f"  [output] {resultado}")
            return None
        raise RuntimeError(f"Nodo desconocido: {type(nodo)}")

# Test completo: Lexer → Parser → Intérprete
print("Ejecutando programa:")
print("-" * 40)
interprete = Interprete()
ast = Parser(Lexer(codigo).tokenizar()).parsear()
interprete.ejecutar(ast)
print("-" * 40)
print(f"Variables finales: {interprete.env}")

## ✏️ Ejercicio 1: Añadir soporte para `while`

**Problema:** Extiende el lexer, parser e intérprete para soportar:
```
let i = 0
while i < 5
    print(i)
    let i = i + 1
end
```

In [ ]:
# ✏️ EJERCICIO 1

# TODO: 
# 1. Añade TipoToken.WHILE a la enum y PALABRAS_CLAVE
# 2. Crea NodoWhile(condicion, cuerpo)
# 3. En Parser._sentencia(): si t.tipo == WHILE → _while_stmt()
# 4. En Interprete._evaluar(): manejar NodoWhile con un bucle Python

# Código de prueba:
codigo_while = """
let i = 0
while i < 3
    print(i)
    let i = i + 1
end
"""
# Tu implementación aquí...
print("TODO: Implementar while")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1

# 1. Nuevo tipo de token
TipoToken.WHILE = TipoToken.LET   # Reutilizamos valor, clase ya no es mutable

# Mejor: creamos subclases extendidas
from enum import Enum, auto
class TipoTokenV2(Enum):
    NUMERO=auto(); BOOLEANO=auto(); IDENTIFICADOR=auto()
    MAS=auto(); MENOS=auto(); ESTRELLA=auto(); BARRA=auto()
    ASIGNAR=auto(); IGUAL=auto(); NO_IGUAL=auto(); MENOR=auto(); MAYOR=auto()
    PAREN_IZQ=auto(); PAREN_DER=auto(); COMA=auto(); NUEVA_LINEA=auto()
    LET=auto(); IF=auto(); ELSE=auto(); END=auto(); PRINT=auto()
    TRUE=auto(); FALSE=auto(); WHILE=auto(); EOF=auto()

PALABRAS_CLAVE_V2 = {**PALABRAS_CLAVE, "while": TipoTokenV2.WHILE}

@dataclass
class NodoWhile:
    condicion: object
    cuerpo: list

class InterpreteConWhile(Interprete):
    def _evaluar(self, nodo):
        if isinstance(nodo, NodoWhile):
            while self._evaluar(nodo.condicion):
                for s in nodo.cuerpo:
                    self._evaluar(s)
            return None
        return super()._evaluar(nodo)

# Simulamos manualmente (sin re-escribir el lexer completo)
ast_while = [
    NodoLet("i", NodoNumero(0)),
    NodoWhile(
        NodoBinario("<", NodoIdentificador("i"), NodoNumero(3)),
        [
            NodoPrint([NodoIdentificador("i")]),
            NodoLet("i", NodoBinario("+", NodoIdentificador("i"), NodoNumero(1))),
        ]
    )
]

interp_while = InterpreteConWhile()
print("Ejecutando while (i de 0 a 2):")
interp_while.ejecutar(ast_while)

## 🎯 Quiz — Preguntas de Entrevista

**P1:** ¿Cuál es la diferencia entre un compilador y un intérprete?
> **R:** El compilador traduce el código fuente a código máquina/bytecode ANTES de ejecutar.
> El intérprete ejecuta el código directamente (o primero a una representación intermedia).
> Python compila a bytecode (.pyc) y luego lo interpreta con CPython VM.

**P2:** ¿Qué es la recursión mutua y cuándo aparece en parsers?
> **R:** Dos funciones que se llaman mutuamente: `expr()` → `term()` → `factor()` → `expr()`.
> Aparece en gramáticas con expresiones anidadas como `1 + (2 * (3 + 4))`.

**P3:** ¿Por qué separamos el Lexer del Parser?
> **R:** Separación de responsabilidades. El Lexer maneja detalles léxicos (espacios, comentarios).
> El Parser se enfoca en la gramática. También permite reutilizar el Lexer con diferentes parsers.

**P4:** ¿Qué es un AST y qué ventaja tiene sobre el parse tree?
> **R:** El AST elimina nodos intermedios que solo existen por la gramática (ej: factores de una sola
> expresión). Es más compacto y fácil de procesar para análisis, optimización y generación de código.